In [1]:
## load libraries
import os
from dotenv import load_dotenv
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# LangChain core imports
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_core.runnables import (
    RunnablePassthrough, 
 
)
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage, AIMessage

# LangChain specific imports
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import TextLoader, PyPDFLoader
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains import create_retrieval_chain

In [2]:
sample_documents = [
    Document(
        page_content="""
        Artificial Intelligence (AI) is the simulation of human intelligence in machines.
        These systems are designed to think like humans and mimic their actions.
        AI can be categorized into narrow AI and general AI.
        """,
        metadata={"source": "AI Introduction", "page": 1, "topic": "AI"}
    ),
    Document(
        page_content="""
        Machine Learning is a subset of AI that enables systems to learn from data.
        Instead of being explicitly programmed, ML algorithms find patterns in data.
        Common types include supervised, unsupervised, and reinforcement learning.
        """,
        metadata={"source": "ML Basics", "page": 1, "topic": "ML"}
    ),
    Document(
        page_content="""
        Deep Learning is a subset of machine learning based on artificial neural networks.
        It uses multiple layers to progressively extract higher-level features from raw input.
        Deep learning has revolutionized computer vision, NLP, and speech recognition.
        """,
        metadata={"source": "Deep Learning", "page": 1, "topic": "DL"}
    ),
    Document(
        page_content="""
        Natural Language Processing (NLP) is a branch of AI that helps computers understand human language.
        It combines computational linguistics with machine learning and deep learning models.
        Applications include chatbots, translation, sentiment analysis, and text summarization.
        """,
        metadata={"source": "NLP Overview", "page": 1, "topic": "NLP"}
    )
]

print(sample_documents)

[Document(metadata={'source': 'AI Introduction', 'page': 1, 'topic': 'AI'}, page_content='\n        Artificial Intelligence (AI) is the simulation of human intelligence in machines.\n        These systems are designed to think like humans and mimic their actions.\n        AI can be categorized into narrow AI and general AI.\n        '), Document(metadata={'source': 'ML Basics', 'page': 1, 'topic': 'ML'}, page_content='\n        Machine Learning is a subset of AI that enables systems to learn from data.\n        Instead of being explicitly programmed, ML algorithms find patterns in data.\n        Common types include supervised, unsupervised, and reinforcement learning.\n        '), Document(metadata={'source': 'Deep Learning', 'page': 1, 'topic': 'DL'}, page_content='\n        Deep Learning is a subset of machine learning based on artificial neural networks.\n        It uses multiple layers to progressively extract higher-level features from raw input.\n        Deep learning has revolu

In [3]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=20, length_function=len, separators=[" "])

chunks = text_splitter.split_documents(sample_documents)

print(f"Number of chunks created: {len(chunks)}")

Number of chunks created: 8


In [4]:
print(f"Created {len(chunks)} chunks from {len(sample_documents)} documents")
print("\nExample chunk:")
print(f"Content: {chunks[0].page_content}")
print(f"Metadata: {chunks[0].metadata}")

Created 8 chunks from 4 documents

Example chunk:
Content: Artificial Intelligence (AI) is the simulation of human intelligence in machines.
        These systems are designed to think like humans and mimic their actions.
        AI can be
Metadata: {'source': 'AI Introduction', 'page': 1, 'topic': 'AI'}


In [5]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

sample_text = "What is machine learning"
sample_embedding = embeddings.embed_query(sample_text)
print(sample_embedding[:10])  # print first 10 values for brevity

C:\Users\itsar\AppData\Local\Temp\ipykernel_51744\3170782392.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5647.31it/s]


[-0.0290356632322073, 0.0075597334653139114, 0.04076480492949486, 0.030535517260432243, 0.0517570786178112, -0.017347536981105804, -0.03094184398651123, -0.06556293368339539, -0.033060770481824875, -0.009561242535710335]


In [6]:
texts=["AI","Machine learning","Deep Learning","Neural Network"]
batch_embeddings=embeddings.embed_documents(texts)
print(batch_embeddings[0])

[-0.03653926029801369, -0.015164357610046864, 0.016432464122772217, 0.010568826459348202, 0.006010559853166342, -0.018473288044333458, 0.08546527475118637, 0.02096845768392086, 0.027815353125333786, 0.012431694194674492, -0.02937648445367813, -0.031135285273194313, 0.03491251543164253, -0.018150851130485535, -0.06498481333255768, 0.0516824871301651, -0.019606219604611397, -0.015734203159809113, -0.13371676206588745, -0.09645991772413254, -0.02547178603708744, -0.0014895843341946602, -0.006349374074488878, -0.02582065388560295, -0.02737179957330227, 0.12268992513418198, -0.007792469579726458, -0.03852269425988197, 0.014383504167199135, -0.09218426793813705, 0.008695731870830059, 0.00261337636038661, 0.09103471785783768, -0.030313612893223763, -0.09604638814926147, 0.022289087995886803, -0.09024307876825333, -0.032947368919849396, 0.0715833380818367, -0.008893106132745743, -0.025708934292197227, -0.0791396051645279, 0.014530384913086891, -0.07420430332422256, 0.08045009523630142, 0.07804

In [7]:
### Compare Embedding using cosine similarity

def compare_embeddings(text1:str,text2:str):
    """Compare semantic simialrity of 2 texts usign embeddings"""

    emb1=np.array(embeddings.embed_query(text1))
    emb2=np.array(embeddings.embed_query(text2))

    ## Calculate the simialrity score

    similarity=np.dot(emb1, emb2) / (np.linalg.norm(emb1) * np.linalg.norm(emb2))
    return similarity

In [8]:
# Test semantic similarity
print("\nSemantic Similarity Examples:")
print(f"'AI' vs 'Artificial Intelligence': {compare_embeddings('AI', 'Artificial Intelligence'):.3f}")


Semantic Similarity Examples:
'AI' vs 'Artificial Intelligence': 0.791


In [9]:
print(f"'AI' vs 'Pizza': {compare_embeddings('AI', 'Pizza'):.3f}")

'AI' vs 'Pizza': 0.257


In [10]:
print(f"'Machine Learning' vs 'ML': {compare_embeddings('Machine Learning', 'ML'):.3f}")

'Machine Learning' vs 'ML': 0.373


In [11]:
vectorstore=FAISS.from_documents(
    documents=chunks,
    embedding=embeddings
)
print(f"Vector store created with {vectorstore.index.ntotal} vectors")

Vector store created with 8 vectors


In [12]:
vectorstore.save_local("faiss_index")
print("Vector store saved to 'faiss_index' directory")

Vector store saved to 'faiss_index' directory


In [13]:
## load vector store
loaded_vectorstore=FAISS.load_local(
    "faiss_index",
    embeddings,
    allow_dangerous_deserialization=True
)

print(f"Loaded vector store contains {loaded_vectorstore.index.ntotal} vectors")

Loaded vector store contains 8 vectors


In [14]:
## Similarity Search 
query="What is deep learning"

results=vectorstore.similarity_search(query,k=3)
print(results)

[Document(id='568dcc1d-d62b-44ec-a974-4f057d2c5d5b', metadata={'source': 'Deep Learning', 'page': 1, 'topic': 'DL'}, page_content='Deep Learning is a subset of machine learning based on artificial neural networks.\n        It uses multiple layers to progressively extract higher-level features from raw input.\n        Deep'), Document(id='0581cacb-c010-4ee9-96de-51ec4072c48a', metadata={'source': 'Deep Learning', 'page': 1, 'topic': 'DL'}, page_content='input.\n        Deep learning has revolutionized computer vision, NLP, and speech recognition.'), Document(id='bebf2cb6-d524-495d-a038-b4c0b19380b6', metadata={'source': 'NLP Overview', 'page': 1, 'topic': 'NLP'}, page_content='Natural Language Processing (NLP) is a branch of AI that helps computers understand human language.\n        It combines computational linguistics with machine learning and deep learning')]


In [15]:
print(f"Query: {query}\n")
print("Top 3 similar chunks:")
for i, doc in enumerate(results):
    print(f"\n{i+1}. Source: {doc.metadata['source']}")
    print(f"   Content: {doc.page_content[:200]}...")

Query: What is deep learning

Top 3 similar chunks:

1. Source: Deep Learning
   Content: Deep Learning is a subset of machine learning based on artificial neural networks.
        It uses multiple layers to progressively extract higher-level features from raw input.
        Deep...

2. Source: Deep Learning
   Content: input.
        Deep learning has revolutionized computer vision, NLP, and speech recognition....

3. Source: NLP Overview
   Content: Natural Language Processing (NLP) is a branch of AI that helps computers understand human language.
        It combines computational linguistics with machine learning and deep learning...


In [16]:
### Similarity Search with score
results_with_scores=vectorstore.similarity_search_with_score(query,k=3)

print(f"Query: {query}\n")
print("Top 3 similar chunks:")
print("\n\nSimilarity search with scores:")
for doc, score in results_with_scores:
    print(f"\nScore: {score:.3f}")
    print(f"Source: {doc.metadata['source']}")
    print(f"Content preview: {doc.page_content[:100]}...")

Query: What is deep learning

Top 3 similar chunks:


Similarity search with scores:

Score: 0.459
Source: Deep Learning
Content preview: Deep Learning is a subset of machine learning based on artificial neural networks.
        It uses m...

Score: 0.675
Source: Deep Learning
Content preview: input.
        Deep learning has revolutionized computer vision, NLP, and speech recognition....

Score: 0.991
Source: NLP Overview
Content preview: Natural Language Processing (NLP) is a branch of AI that helps computers understand human language.
...


In [17]:
### Search with metadata filtering
filter_dict={"topic":"ML"}
filtered_results=vectorstore.similarity_search(
    query,
    k=3,
    filter=filter_dict
)
print(filtered_results)
len(filtered_results)

[Document(id='3dc2ed5a-c392-4577-ae79-1dcf8ebb7113', metadata={'source': 'ML Basics', 'page': 1, 'topic': 'ML'}, page_content='Machine Learning is a subset of AI that enables systems to learn from data.\n        Instead of being explicitly programmed, ML algorithms find patterns in data.\n        Common types include'), Document(id='d789003c-ddfa-4263-8e36-ff01496100b1', metadata={'source': 'ML Basics', 'page': 1, 'topic': 'ML'}, page_content='types include supervised, unsupervised, and reinforcement learning.')]


2

In [22]:
import os
from langchain_groq import ChatGroq
from langchain.chat_models import init_chat_model

# Make sure your API key is set correctly
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

# Use the most stable, globally available model
llm = init_chat_model(model="groq:openai/gpt-oss-120b")

response = llm.invoke("Hi")
print(response.content)


Hello! How can I assist you today?


In [28]:
# 1. Simple RAG Chain with LCEL
simple_prompt = ChatPromptTemplate.from_template("""Answer the question based only on the following context:
Context: {context}

Question: {question}

Answer:""")

In [29]:
## Basic retriever
retriever=vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k":3}
)

In [30]:
from typing import List
# Format documents for the prompt
def format_docs(docs: List[Document]) -> str:
    """Format documents for insertion into prompt"""
    formatted = []
    for i, doc in enumerate(docs):
        source = doc.metadata.get('source', 'Unknown') # the 'Unknown' is simply a default value.
        formatted.append(f"Document {i+1} (Source: {source}):\n{doc.page_content}")
    return "\n\n".join(formatted) # List -> String


# Test format_docs function
test_docs = [
    Document(page_content="This is the content of the first document.", metadata={"source": "doc1"}),
    Document(page_content="This is the content of the second document.", metadata={"source": "doc2"})
]
print(format_docs(test_docs))

Document 1 (Source: doc1):
This is the content of the first document.

Document 2 (Source: doc2):
This is the content of the second document.


In [36]:
simple_rag_chain=(
    {"context":retriever | format_docs,"question":RunnablePassthrough() } # RunnablePassthrough() → just forwards the original user input unchanged, so it becomes the question.
    | simple_prompt
    | llm
    | StrOutputParser() # StrOutputParser is a simple output parser in LangChain that takes the raw response from the LLM and converts it into a plain Python string.

)
simple_rag_chain

{
  context: VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000001C186F1EA50>, search_kwargs={'k': 3})
           | RunnableLambda(format_docs),
  question: RunnablePassthrough()
}
| ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='Answer the question based only on the following context:\nContext: {context}\n\nQuestion: {question}\n\nAnswer:'), additional_kwargs={})])
| ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.1', 'langchain': '1.3.18'}}, profile={'name': 'GPT OSS 120B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': Fals

In [37]:
### Conversational RAg Chain

conversational_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful AI assistant. Use the provided context to answer questions."),
    ("placeholder", "{chat_history}"),
    ("human", "Context: {context}\n\nQuestion: {input}"),
])

In [38]:
def create_conversational_rag():
    """Create a conversational RAG chain with memory"""
    return (
        RunnablePassthrough.assign(
            context=lambda x: format_docs(retriever.invoke(x["input"]))
        )
        | conversational_prompt
        | llm
        | StrOutputParser()
    )

conversational_rag = create_conversational_rag()
conversational_rag

RunnableAssign(mapper={
  context: RunnableLambda(lambda x: format_docs(retriever.invoke(x['input'])))
})
| ChatPromptTemplate(input_variables=['context', 'input'], optional_variables=['chat_history'], input_types={'chat_history': list[typing.Annotated[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')] | typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')] | typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')] | typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')] | typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')] | typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')] | typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')] | typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')] | typing.Annotated[langchain_core.messages.chat.ChatMessageChunk, Tag(tag='Chat

In [39]:
### streaming RAG chain
streaming_rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | simple_prompt
    | llm
)

print("Modern RAG chains created successfully!")
print("Available chains:")
print("- simple_rag_chain: Basic Q&A")
print("- conversational_rag: Maintains conversation history")
print("- streaming_rag_chain: Supports token streaming")

Modern RAG chains created successfully!
Available chains:
- simple_rag_chain: Basic Q&A
- conversational_rag: Maintains conversation history
- streaming_rag_chain: Supports token streaming


In [41]:
def test_rag_chains(question: str):
    """Test all RAG chain variants"""
    print(f"Question: {question}")
    print("=" * 80)

    print("\n1. Simple RAG Chain:")
    answer = simple_rag_chain.invoke(question)
    print(f"Answer: {answer}")

    print("\n2. Streaming RAG:")
    print("Answer: ", end="", flush=True)
    for chunk in streaming_rag_chain.stream(question):
        print(chunk.content, end="", flush=True)
    print()

    print("\n3. Conversational RAG Chain:")
    chat_history = []
    for i in range(2):
        user_input = input(f"User (Round {i+1}): ")
        chat_history.append(HumanMessage(content=user_input))
        response = conversational_rag.invoke({"input": user_input, "chat_history": chat_history})
        print(f"AI: {response}")
        chat_history.append(AIMessage(content=response))

test_rag_chains("What is the difference between AI and machine learning")

Question: What is the difference between AI and machine learning

1. Simple RAG Chain:
Answer: AI (Artificial Intelligence) is the broader field that involves creating machines that simulate human intelligence and can think and act like humans. Machine learning is a subset of AI; it specifically refers to techniques that enable systems to learn from data and discover patterns without being explicitly programmed. In short, AI is the overall goal of intelligent behavior, while machine learning is one way to achieve that goal by using data‑driven learning.

2. Streaming RAG:
Answer: **AI vs. Machine Learning (based on the provided context)**  

- **Artificial Intelligence (AI)** – Defined as *the simulation of human intelligence in machines*. AI systems are built to *think like humans* and *mimic human actions* (Document 2). AI is a broad field that encompasses any technique that enables a machine to exhibit intelligent behavior.

- **Machine Learning (ML)** – Described as *a subset of AI